# Exploratory Retention Prioritization Simulation

Optional development-only simulation comparing random, risk-only, value-proxy-only and risk×value rankings at 10%, 20% and 30% capacity. It is explicitly non-causal and non-operational.

## Development-only ranking simulation

In [1]:
from pathlib import Path
import json, yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss, precision_recall_curve, f1_score

ROOT = Path.cwd().resolve()
if not (ROOT / "configs" / "analysis_registry.yaml").exists():
    for parent in ROOT.parents:
        if (parent / "configs" / "analysis_registry.yaml").exists():
            ROOT = parent
            break
registry = yaml.safe_load((ROOT / "configs" / "analysis_registry.yaml").read_text())
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.models.ebm_experiments import load_formal_ebm_input
from src.models.config import load_model_config

frames = {}
for model, meta in registry["models"].items():
    if model == "EBM":
        ebm_config = load_model_config(ROOT / "configs/models/ebm.yaml", strict_schema=True)
        if meta["oof_path"] != ebm_config["canonical_results"]["oof_predictions"]:
            raise ValueError("Group EBM registry must point to the configured formal OOF input.")
        d, ebm_fold_metrics, ebm_summary = load_formal_ebm_input(ROOT)
    else:
        d = pd.read_csv(ROOT / meta["oof_path"])
    d = d.sort_values("Employee_ID").reset_index(drop=True)
    frames[model] = d
print(f"Loaded {len(frames)} headline-model OOF files from development data only.")
from src.prioritization.value_proxy import build_value_proxy
from src.prioritization.ranking import evaluate_ranking
comparison=pd.read_csv(ROOT/"results/comparison/model_comparison.csv").sort_values("pooled_average_precision",ascending=False)
leader=str(comparison.iloc[0].model)
raw=pd.read_csv(ROOT/"data/raw/Extended_Employee_Performance_and_Productivity_Data.csv")
reg=pd.read_csv(ROOT/"data/processed/split_registry.csv")
dev=raw.merge(reg[["Employee_ID","partition"]],on="Employee_ID"); dev=dev[dev.partition=="development"].drop(columns="partition")
risk=frames[leader][["Employee_ID","y_true","y_probability"]].merge(dev,on="Employee_ID",validate="one_to_one")
risk["value_proxy"]=build_value_proxy(risk); risk["risk_value_score"]=risk.y_probability*risk.value_proxy
rng=np.random.default_rng(42); risk["random_score"]=rng.random(len(risk))
rows=[]
for cap in [.10,.20,.30]:
    for strategy,score in [("random","random_score"),("risk_only","y_probability"),("value_proxy_only","value_proxy"),("risk_x_value","risk_value_score")]:
        result=evaluate_ranking(risk,score,cap); result.update({"strategy":strategy,"risk_model":leader}); rows.append(result)
summary=pd.DataFrame(rows); out=ROOT/"results/prioritization"; out.mkdir(parents=True,exist_ok=True); summary.to_csv(out/"prioritization_summary.csv",index=False)
decision={"status":"exploratory development-only simulation","risk_model":leader,"risk_model_finally_frozen":False,"calibration_used":False,"warning":"Not an operational recommendation. The risk model remains near no-skill and no final model/calibrator is frozen."}; (out/"prioritization_decision.json").write_text(json.dumps(decision,indent=2))
display(summary.round(5)); decision

Loaded 6 headline-model OOF files from development data only.


,capacity,selected_count,captured_positive_count,precision_at_capacity,recall_at_capacity,lift_at_capacity,strategy,risk_model
0,0.1,8000,773,0.09662,0.09653,0.96528,random,XGBoost
1,0.1,8000,828,0.10350,0.10340,1.03397,risk_only,XGBoost
2,0.1,8000,788,0.09850,0.09840,0.98402,value_proxy_only,XGBoost
3,0.1,8000,816,0.10200,0.10190,1.01898,risk_x_value,XGBoost
4,0.2,16000,1551,0.09694,0.19368,0.96841,random,XGBoost
5,0.2,16000,1632,0.10200,0.20380,1.01898,risk_only,XGBoost
6,0.2,16000,1603,0.10019,0.20017,1.00087,value_proxy_only,XGBoost
7,0.2,16000,1581,0.09881,0.19743,0.98714,risk_x_value,XGBoost
8,0.3,24000,2354,0.09808,0.29396,0.97985,random,XGBoost
9,0.3,24000,2411,0.10046,0.30107,1.00358,risk_only,XGBoost


{'status': 'exploratory development-only simulation',
 'risk_model': 'XGBoost',
 'risk_model_finally_frozen': False,
 'calibration_used': False,
 'warning': 'Not an operational recommendation. The risk model remains near no-skill and no final model/calibrator is frozen.'}

## Conclusion

Because the predictive model is close to no-skill, this stage is retained only as an academic decision-simulation exercise. It must not be presented as evidence that interventions would work or that employees should be targeted in practice.